# ProjectMem Kaggle Full Runner

Notebook nay chay full benchmark suite bang `app/main.py` truc tiep tren Kaggle.

- Mac dinh dung `Qwen/Qwen2.5-1.5B-Instruct` cho ca hai agent de giam rui ro OOM
- `memae` de ngoai mac dinh vi can parquet local
- Output duoc luu trong `/kaggle/working/projectmem_reports_full`


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/Dung205789/MultiAgentConflictResolution.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/ProjectMem"

if os.path.isdir(WORKDIR):
    subprocess.run(["git", "-C", WORKDIR, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", WORKDIR, "checkout", BRANCH], check=False)
    subprocess.run(["git", "-C", WORKDIR, "pull", "origin", BRANCH], check=False)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
print("Working directory:", os.getcwd())
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip())


In [ ]:
%cd /kaggle/working/ProjectMem
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt

import os
from huggingface_hub import login

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face login complete.")
else:
    print("HF_TOKEN not set. Public benchmarks and models can still download if available.")


In [ ]:
import os
import torch

USE_DUMMY = False
ALLOW_DUMMY_FALLBACK = True
MAX_SCENARIOS = None
DEVICE = "auto"
OUTPUT_ROOT = "/kaggle/working/projectmem_reports_full"

MODEL_NAME1 = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_NAME2 = "Qwen/Qwen2.5-1.5B-Instruct"

INCLUDE_LOCAL_MEMAE = False
MEMAE_LOCAL_PATH = "data/raw/memab/Conflict_Resolution-00000-of-00001.parquet"

BENCHMARK_SPECS = [
    {"benchmark": "mab_conflict"},
    {"benchmark": "mab", "subset": "all"},
    {"benchmark": "longmemeval", "subset": "oracle"},
    {"benchmark": "safeflow"},
    {"benchmark": "locomo", "subset": "all"}
]

if INCLUDE_LOCAL_MEMAE:
    BENCHMARK_SPECS.insert(0, {"benchmark": "memae"})

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Agent 1 model:", MODEL_NAME1)
print("Agent 2 model:", MODEL_NAME2)
print("Allow dummy fallback:", ALLOW_DUMMY_FALLBACK)
print("Benchmark specs:", BENCHMARK_SPECS)


In [ ]:
import json
import shlex
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

output_root = Path(OUTPUT_ROOT)
summary = {
    "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "benchmarks": [],
    "use_dummy": USE_DUMMY,
    "device": DEVICE,
    "agent1_model": None if USE_DUMMY else MODEL_NAME1,
    "agent2_model": None if USE_DUMMY else MODEL_NAME2,
    "max_scenarios": MAX_SCENARIOS,
    "allow_dummy_fallback": ALLOW_DUMMY_FALLBACK,
}

if INCLUDE_LOCAL_MEMAE and not Path(MEMAE_LOCAL_PATH).exists():
    print(f"WARNING: {MEMAE_LOCAL_PATH} not found. memae run will fail unless you add that parquet first.")

for spec in BENCHMARK_SPECS:
    benchmark = spec["benchmark"]
    run_name = benchmark if "subset" not in spec else f"{benchmark}_{spec['subset']}"
    output_dir = output_root / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "app/main.py",
        "--benchmark", benchmark,
        "--output-dir", str(output_dir),
        "--device", DEVICE,
    ]

    if MAX_SCENARIOS is not None:
        cmd.extend(["--max-scenarios", str(MAX_SCENARIOS)])
    if "subset" in spec:
        cmd.extend(["--subset", spec["subset"]])

    if USE_DUMMY:
        cmd.append("--use-dummy")
    else:
        cmd.extend([
            "--agent1-model", MODEL_NAME1,
            "--agent2-model", MODEL_NAME2,
        ])

    print("\n" + "=" * 72)
    print(f"Running benchmark via app/main.py: {run_name}")
    print(shlex.join(cmd))
    print("=" * 72)

    completed = subprocess.run(cmd, check=False)
    fallback_completed = None

    if completed.returncode != 0 and not USE_DUMMY and ALLOW_DUMMY_FALLBACK:
        fallback_cmd = [
            sys.executable,
            "app/main.py",
            "--benchmark", benchmark,
            "--output-dir", str(output_dir),
            "--device", DEVICE,
            "--use-dummy",
        ]
        if MAX_SCENARIOS is not None:
            fallback_cmd.extend(["--max-scenarios", str(MAX_SCENARIOS)])
        if "subset" in spec:
            fallback_cmd.extend(["--subset", spec["subset"]])

        print("\n" + "-" * 72)
        print(f"Primary run failed for {run_name}. Retrying in dummy mode.")
        print(shlex.join(fallback_cmd))
        print("-" * 72)
        fallback_completed = subprocess.run(fallback_cmd, check=False)

    effective = fallback_completed or completed
    summary["benchmarks"].append({
        "benchmark": benchmark,
        "run_name": run_name,
        "subset": spec.get("subset"),
        "returncode": effective.returncode,
        "status": "success" if effective.returncode == 0 else "failed",
        "primary_returncode": completed.returncode,
        "used_dummy_fallback": fallback_completed is not None,
        "fallback_returncode": None if fallback_completed is None else fallback_completed.returncode,
        "output_dir": str(output_dir),
    })

summary_path = output_root / "matrix_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved summary:", summary_path)


In [ ]:
import zipfile
from pathlib import Path

output_root = Path(OUTPUT_ROOT)
for path in sorted(output_root.rglob("*.json")):
    print(path)

zip_path = Path("/kaggle/working/projectmem_kaggle_full_reports.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in output_root.rglob("*"):
        if file_path.is_file():
            archive.write(file_path, file_path.relative_to(output_root.parent))

print("Created:", zip_path)
